# DataCite Resolution Logs — Athena Analysis

Queries against `datacite.resolution_logs` in AWS Athena (us-east-2).  
Data: May 2026 — 445,233,913 resolution events across 4 AWS regions.

In [ ]:
!pip install pyathena pandas -q

In [ ]:
import pyathena
import pandas as pd

AWS_ACCESS_KEY_ID     = 'YOUR_AWS_ACCESS_KEY_ID'
AWS_SECRET_ACCESS_KEY = 'YOUR_AWS_SECRET_ACCESS_KEY'
AWS_REGION            = 'us-east-2'
S3_STAGING_DIR        = 's3://datacite-logs-processed/athena-results/'

conn = pyathena.connect(
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    s3_staging_dir=S3_STAGING_DIR,
    region_name=AWS_REGION,
)

def query(sql):
    return pd.read_sql(sql, conn)

print('Connected to Athena ✓')

## Row counts by region

In [ ]:
query("""
SELECT region, count(*) AS rows
FROM datacite.resolution_logs
WHERE year=2026 AND month=5
GROUP BY region
ORDER BY rows DESC
""")

## Response code distribution

In [ ]:
# Handle System response codes:
#   1   = Success (DOI resolved)
#   100 = Handle Not Found (DOI doesn't exist)
#   200 = Values Not Found (DOI exists, no URL)
#   2   = Error
query("""
SELECT response_code, count(*) AS cnt
FROM datacite.resolution_logs
WHERE year=2026 AND month=5
GROUP BY response_code
ORDER BY cnt DESC
""")

## Unique DOIs per response code

In [ ]:
query("""
SELECT response_code, count(DISTINCT doi) AS unique_dois
FROM datacite.resolution_logs
WHERE year=2026 AND month=5
GROUP BY response_code
ORDER BY unique_dois DESC
""")

## Successful resolutions (response_code = 1)

In [ ]:
query("""
SELECT
  count(*)              AS total_resolutions,
  count(DISTINCT doi)   AS unique_dois
FROM datacite.resolution_logs
WHERE year=2026 AND month=5
AND response_code = 1
""")

## Top 100 referrer URLs

In [ ]:
query("""
SELECT referrer_url, count(*) AS cnt
FROM datacite.resolution_logs
WHERE year=2026 AND month=5
GROUP BY referrer_url
ORDER BY cnt DESC
LIMIT 100
""")

## AI tool referrer traffic

In [ ]:
df_ai = query("""
SELECT referrer_url, count(*) AS cnt
FROM datacite.resolution_logs
WHERE year=2026 AND month=5
  AND (
    referrer_url LIKE '%chatgpt.com%'
    OR referrer_url LIKE '%gemini.google.com%'
    OR referrer_url LIKE '%elicit.com%'
    OR referrer_url LIKE '%consensus.app%'
    OR referrer_url LIKE '%scispace.com%'
    OR referrer_url LIKE '%researchrabbit.ai%'
    OR referrer_url LIKE '%doubao.com%'
    OR referrer_url LIKE '%bohrium.com%'
    OR referrer_url LIKE '%perplexity.ai%'
    OR referrer_url LIKE '%claude.ai%'
    OR referrer_url LIKE '%copilot.microsoft.com%'
    OR referrer_url LIKE '%you.com%'
    OR referrer_url LIKE '%phind.com%'
    OR referrer_url LIKE '%poe.com%'
  )
GROUP BY referrer_url
ORDER BY cnt DESC
""")

TOTAL = 445_233_913
df_ai['pct_of_total'] = (df_ai['cnt'] / TOTAL * 100).round(4)
print(f"Total AI traffic: {df_ai['cnt'].sum():,}  ({df_ai['cnt'].sum()/TOTAL*100:.3f}% of all traffic)")
df_ai

## User agent breakdown

In [ ]:
import re
from collections import defaultdict

df_ua = query("""
SELECT user_agent, count(*) AS cnt
FROM datacite.resolution_logs
WHERE year=2026 AND month=5
GROUP BY user_agent
ORDER BY cnt DESC
LIMIT 2000
""")

def classify(ua):
    if not ua or pd.isna(ua):
        return 'Unknown / Empty'
    u = ua.lower()
    if 'linerbot' in u: return 'LinerBot'
    if any(x in u for x in ['googlebot', 'google-read-aloud', 'adsbot-google']): return 'Googlebot'
    if any(x in u for x in ['bingbot', 'msnbot']): return 'Bingbot'
    if 'yandexbot' in u: return 'Yandexbot'
    if 'baiduspider' in u: return 'Baiduspider'
    if 'semrushbot' in u: return 'SEMrushBot'
    if 'ahrefsbot' in u: return 'AhrefsBot'
    if 'petalbot' in u: return 'PetalBot'
    if 'facebookexternalhit' in u: return 'Facebook Bot'
    if 'gptbot' in u or 'chatgpt-user' in u: return 'GPTBot (OpenAI)'
    if 'claudebot' in u or 'anthropic' in u: return 'ClaudeBot (Anthropic)'
    if 'perplexitybot' in u: return 'PerplexityBot'
    if any(x in u for x in ['scrapy', 'crawler', 'spider']): return 'Other Crawlers'
    if re.search(r'bot[/ \)]', u): return 'Other Bots'
    if 'curl/' in u: return 'curl'
    if 'python-requests' in u: return 'Python (requests)'
    if 'aiohttp' in u: return 'Python (aiohttp)'
    if 'httpx' in u: return 'Python (httpx)'
    if re.search(r'\bpython\b', u): return 'Python (other)'
    if any(x in u for x in ['httr', 'rcurl', 'rvest']): return 'R'
    if 'java/' in u or 'apache-httpclient' in u or 'okhttp' in u: return 'Java'
    if 'go-http-client' in u: return 'Go'
    if 'ruby' in u: return 'Ruby'
    if 'axios' in u or 'node-fetch' in u: return 'Node.js'
    if 'wget' in u: return 'wget'
    if 'edg/' in u or 'edge/' in u: return 'Microsoft Edge'
    if 'opr/' in u or 'opera' in u: return 'Opera'
    if 'samsungbrowser' in u: return 'Samsung Browser'
    if 'firefox/' in u or 'fxios' in u: return 'Firefox'
    if 'chrome/' in u or 'crios/' in u or 'chromium' in u: return 'Chrome'
    if 'safari/' in u: return 'Safari'
    if 'msie' in u or 'trident/' in u: return 'Internet Explorer'
    return 'Other'

df_ua['category'] = df_ua['user_agent'].apply(classify)
summary = df_ua.groupby('category')['cnt'].sum().reset_index()
summary['pct_of_total'] = (summary['cnt'] / TOTAL * 100).round(3)
summary.sort_values('cnt', ascending=False).reset_index(drop=True)